<a href="https://colab.research.google.com/github/lauernling07-cell/CVLP_Emotion_Detection-/blob/main/Important.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Automatically mounting the Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from bs4 import BeautifulSoup
import string
import re

stopwords = [
    "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any",
    "are", "aren't", "as", "at", "be", "because", "been", "before", "being", "below",
    "between", "both", "but", "by", "can't", "cannot", "could", "couldn't", "did",
    "didn't", "do", "does", "doesn't", "doing", "don't", "down", "during", "each",
    "few", "for", "from", "further", "had", "hadn't", "has", "hasn't", "have", "haven't",
    "having", "he", "he'd", "he'll", "he's", "her", "here", "here's", "hers", "herself",
    "him", "himself", "his", "how", "how's", "i", "i'd", "i'll", "i'm", "i've", "if",
    "in", "into", "is", "isn't", "it", "it's", "its", "itself", "let's", "me", "more",
    "most", "mustn't", "my", "myself", "no", "nor", "not", "of", "off", "on", "once",
    "only", "or", "other", "ought", "our", "ours", "ourselves", "out", "over", "own",
    "same", "shan't", "she", "she'd", "she'll", "she's", "should", "shouldn't", "so",
    "some", "such", "than", "that", "that's", "the", "their", "theirs", "them", "themselves",
    "then", "there", "there's", "these", "they", "they'd", "they'll", "they're", "they've",
    "this", "those", "through", "to", "too", "under", "until", "up", "very", "was",
    "wasn't", "we", "we'd", "we'll", "we're", "we've", "were", "weren't", "what",
    "what's", "when", "when's", "where", "where's", "which", "while", "who", "who's",
    "whom", "why", "why's", "with", "won't", "would", "wouldn't", "you", "you'd",
    "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves"
]

negations = {"not", "no", "nor", "never", "neither", "cannot", "can't", "don't", "doesn't", "didn't", "won't", "wouldn't", "shouldn't", "couldn't",
             "isn't", "aren't", "wasn't", "weren't", "haven't", "hasn't", "hadn't"}

# Function of Removing Stopwords
def remove_stopwords(sentence):
    words = sentence.split()
    filtered_sentence = ""

    for word in words:
        if word.lower() not in stopwords and word.lower() not in negations:
            filtered_sentence += word + " "
    return filtered_sentence.strip()


# Function of Removing the Punctuation
def remove_punctuation(sentence):
    table = str.maketrans('', '', string.punctuation)
    words = sentence.split()
    filtered_sentence = ""

    for word in words:
        word = word.translate(table)
        filtered_sentence += word + " "

    return filtered_sentence.strip()

# Function of removing URL, Mentions and Hashtags
def remove_url_mention_hashtag(text):

  # Remove the URL and Mentions
  text = re.sub(r"https://\S+", '',text)
  text = re.sub(r"www\S+", '',text)
  text = re.sub(r"@\w+", '',text)

  # Remove the Hashtags
  text = text.replace('#', '')

  return text


# Function of cleaning the text for baseline model
def cleanTextBaseLineModel(text):
    text = remove_url_mention_hashtag(text)
    text = remove_punctuation(text)
    text = remove_stopwords(text)

    return text

# Function of cleaning the text for Tuned Model (Transformer)
def cleanTextTunedModel(text):
  text = remove_url_mention_hashtag(text)
  return text




In [4]:
import pandas as pd
import numpy as np

# Load the dataset & remove rows with with incomplete info
path = "/content/drive/MyDrive/CVLP_Datasets/tweet_emotions.csv"
df = pd.read_csv(path)
df = df.dropna(subset=['content', 'sentiment'])

# 2. Extract text and emotion labels
texts = df['content'].values

cleaned_text_baseline = []
cleaned_text_tuned = []

for text in texts:
  cleaned_text = cleanTextBaseLineModel(text)
  cleaned_text_baseline.append(cleaned_text)

for text in texts:
  cleaned_text = cleanTextTunedModel(text)
  cleaned_text_tuned.append(cleaned_text)

# 3. Encode the labels
labels = sorted(df['sentiment'].unique())
label_mapping = {}

for i, label in enumerate(labels):
    label_mapping[label] = i

df['label'] = df['sentiment'].map(label_mapping)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

y = df['label'].values
X_train_base, X_test_base, X_train_dl, X_test_dl, y_train, y_test = train_test_split(
    cleaned_text_baseline,
    cleaned_text_tuned,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# --- FEATURE REPRESENTATION ---
# 1. TF-IDF
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_base)
X_test_tfidf  = tfidf_vectorizer.transform(X_test_base)

# 2. Keras Tokenizer (For LSTM)
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_dl)
train_padded = pad_sequences(tokenizer.texts_to_sequences(X_train_dl), maxlen=100, padding='post', truncating='post')
test_padded  = pad_sequences(tokenizer.texts_to_sequences(X_test_dl), maxlen=100, padding='post', truncating='post')

NLP Model Implementation